# Signature-transform forecasting on public labor-market data

This notebook applies the signature-transform forecasting framework to a fully public weekly dataset built from FRED. The target is **U.S. initial jobless claims** and the covariates are market and macro series that update at daily or weekly frequency. The goal is to test whether the same adaptive signature approach can capture nonlinear path effects and changing economic regimes outside the original freight-rate application.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from marketplace_signature_forecast.config import (
    ALL_HORIZONS,
    DEFAULT_DEPTH,
    DEFAULT_N_TARGET,
    DEFAULT_WINDOW_SIZE,
    FRED_API_KEY,
)
from marketplace_signature_forecast.data_loading import (
    build_data_dictionary_from_specs,
    fetch_series_bundle,
)
from marketplace_signature_forecast.preprocessing import (
    build_model_dataset,
    prepare_standardized_arrays,
    resample_collection,
    resample_to_weekly,
    save_processed_bundle,
    train_validation_split,
)
from marketplace_signature_forecast.adaptive_weights import rolling_adaptive_weights
from marketplace_signature_forecast.modeling import rolling_forecast
from marketplace_signature_forecast.evaluation import (
    build_forecast_dataframe,
    compare_with_baselines,
    inverse_transform_forecasts,
    run_multi_horizon_experiment,
)
from marketplace_signature_forecast.plotting import (
    plot_adaptive_weight_diagnostics,
    plot_correlation_matrix,
    plot_input_grid,
    plot_target_split,
    plot_weights_for_origin,
)

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', None)


## 1. Dataset specification

The target is a weekly labor-market series, and all inputs are fetched from FRED as public daily or weekly series. This makes the dataset easy to rerun without proprietary files.


In [ ]:
START_DATE = '2000-01-01'
END_DATE = '2024-12-31'
RESAMPLE_FREQ = 'W-SAT'
N_VALIDATION = 26
EXPERIMENT_NAME = 'initial_claims_public'

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / EXPERIMENT_NAME
FIGURES_DIR = PROJECT_ROOT / 'figures' / EXPERIMENT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'signature_y_only').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'signature_joint_path').mkdir(parents=True, exist_ok=True)

TARGET = {
    'code': 'ICSA',
    'name': 'initial_claims',
    'description': 'Initial Claims, seasonally adjusted',
    'source': 'FRED',
    'frequency': 'Weekly, Ending Saturday',
}

FRED_INPUTS = {
    'CCSA': {
        'name': 'continued_claims',
        'description': 'Continued Claims (Insured Unemployment)',
        'frequency': 'Weekly, Ending Saturday',
    },
    'NFCI': {
        'name': 'nfci',
        'description': 'Chicago Fed National Financial Conditions Index',
        'frequency': 'Weekly, Ending Friday',
    },
    'VIXCLS': {
        'name': 'vix',
        'description': 'CBOE Volatility Index',
        'frequency': 'Daily, Close',
    },
    'T10Y2Y': {
        'name': 'term_spread',
        'description': '10Y minus 2Y Treasury spread',
        'frequency': 'Daily',
    },
    'SP500': {
        'name': 'sp500',
        'description': 'S&P 500 Index',
        'frequency': 'Daily, Close',
    },
    'DCOILWTICO': {
        'name': 'wti_crude',
        'description': 'WTI crude oil spot price',
        'frequency': 'Daily',
    },
    'EFFR': {
        'name': 'effr',
        'description': 'Effective Federal Funds Rate',
        'frequency': 'Daily',
    },
}

data_dictionary = build_data_dictionary_from_specs(TARGET, FRED_INPUTS)
data_dictionary


## 2. Download, align, and save the weekly dataset

All series are resampled to a common weekly calendar ending on Saturday so that the target and covariates share the same forecasting grid.


In [ ]:
y_raw, x_raw = fetch_series_bundle(
    api_key=FRED_API_KEY,
    target=TARGET,
    start=START_DATE,
    end=END_DATE,
    fred_inputs=FRED_INPUTS,
)

y_weekly = resample_to_weekly(y_raw, freq=RESAMPLE_FREQ)
x_weekly = resample_collection(x_raw, freq=RESAMPLE_FREQ)
full_data = build_model_dataset(y_weekly, x_weekly).dropna().copy()
train_data, validation_data = train_validation_split(full_data, N_VALIDATION)

save_processed_bundle(full_data, train_data, validation_data, data_dictionary, OUTPUT_DIR)

print(full_data.shape)
print(full_data.index.min(), full_data.index.max())
full_data.head()


## 3. Exploratory analysis


In [ ]:
plot_target_split(train_data, validation_data, TARGET['name'], FIGURES_DIR / 'target_variable.png')
plot_input_grid(full_data, TARGET['name'], FIGURES_DIR / 'input_variables.png')
plot_correlation_matrix(full_data, FIGURES_DIR / 'correlation_matrix.png')

full_data.describe().round(2)


## 4. Standardized modeling arrays

The forecasting model works on standardized arrays and predicts future changes in the target over multiple horizons.


In [ ]:
prepared = prepare_standardized_arrays(
    full_data=full_data,
    target_col=TARGET['name'],
    horizons=ALL_HORIZONS,
    n_validation=N_VALIDATION,
)

X = prepared['X']
y = prepared['y']
y_raw_array = prepared['y_raw']
dates = prepared['dates']
y_scaler = prepared['y_scaler']
train_end_t = prepared['train_end']

WINDOW_SIZE = DEFAULT_WINDOW_SIZE
DEPTH = DEFAULT_DEPTH
N_TARGET = DEFAULT_N_TARGET
DIAGNOSTIC_HORIZON = 4

print(X.shape, y.shape, train_end_t)


## 5. Adaptive-weight diagnostics

This shows how the signature kernel reweights the historical training windows according to path similarity.


In [ ]:
min_t = WINDOW_SIZE + DIAGNOSTIC_HORIZON + 20
adaptive_results = rolling_adaptive_weights(
    X=X,
    y=y,
    start_t=min_t,
    end_t=train_end_t,
    delta_t=DIAGNOSTIC_HORIZON,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    gamma=None,
    n_target=N_TARGET,
    add_time=True,
)

plot_adaptive_weight_diagnostics(adaptive_results, dates, y, train_end_t, N_TARGET, TARGET['name'])
plot_weights_for_origin(adaptive_results, len(adaptive_results['t']) // 2, X, y, dates, DIAGNOSTIC_HORIZON, WINDOW_SIZE, DEPTH)


## 6. Multi-horizon experiment: signature on the target path only

This mirrors the more conservative specification in which the signature is computed on the recent target trajectory and the contemporaneous external variables enter linearly.


In [ ]:
results_y_only, summary_y_only = run_multi_horizon_experiment(
    X=X,
    y=y,
    dates=dates,
    y_scaler=y_scaler,
    horizons=ALL_HORIZONS,
    n_validation=N_VALIDATION,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    n_target=N_TARGET,
    output_dir=OUTPUT_DIR / 'signature_y_only',
    lambda_lasso=None,
    gamma=None,
    use_sig_y_only=True,
    add_time=True,
)

summary_y_only = summary_y_only.rename(columns={'signature_mre': 'signature_y_only_mre'})
summary_y_only


## 7. Multi-horizon experiment: joint signature of covariates and target

This richer specification feeds the signature transform with the whole local multivariate path, which is useful when the interaction between claims and external variables matters more than the current level of the covariates alone.


In [ ]:
results_joint, summary_joint = run_multi_horizon_experiment(
    X=X,
    y=y,
    dates=dates,
    y_scaler=y_scaler,
    horizons=ALL_HORIZONS,
    n_validation=N_VALIDATION,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    n_target=N_TARGET,
    output_dir=OUTPUT_DIR / 'signature_joint_path',
    lambda_lasso=None,
    gamma=None,
    use_sig_y_only=False,
    add_time=True,
)

summary_joint = summary_joint.rename(columns={'signature_mre': 'signature_joint_path_mre'})
summary_joint


## 8. Compare against naive and ARIMA baselines


In [ ]:
comparison_y_only = compare_with_baselines(
    y_raw=y_raw_array,
    horizons=ALL_HORIZONS,
    n_validation=N_VALIDATION,
    signature_summary=summary_y_only.rename(columns={'signature_y_only_mre': 'signature_mre'}),
).rename(columns={
    'Signature (%)': 'Signature y-only (%)',
    'Improvement vs ARIMA (%)': 'Improvement y-only vs ARIMA (%)',
})

comparison_table = comparison_y_only.merge(
    summary_joint[['horizon_weeks', 'signature_joint_path_mre']],
    left_on='Horizon (weeks)',
    right_on='horizon_weeks',
    how='left',
).drop(columns=['horizon_weeks'])

comparison_table = comparison_table.rename(columns={'signature_joint_path_mre': 'Signature joint-path (%)'})
comparison_table['Improvement joint-path vs ARIMA (%)'] = (
    (comparison_table['ARIMA (%)'] - comparison_table['Signature joint-path (%)'])
    / comparison_table['ARIMA (%)'] * 100
)

comparison_table.to_csv(OUTPUT_DIR / 'comparison_initial_claims.csv', index=False)
comparison_table.round(2)


## 9. Forecast paths for a 4-week horizon

This final view plots the realized validation targets against the two signature specifications for a representative horizon.


In [ ]:
DELTA_T = 4
val_start_t = len(y) - N_VALIDATION - DELTA_T
val_end_t = len(y) - DELTA_T - 1

forecast_y_only = rolling_forecast(
    X=X,
    y=y,
    start_t=val_start_t,
    end_t=val_end_t,
    delta_t=DELTA_T,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    lambda_lasso=None,
    gamma=None,
    n_target=N_TARGET,
    use_sig_y_only=True,
    add_time=True,
)

forecast_joint = rolling_forecast(
    X=X,
    y=y,
    start_t=val_start_t,
    end_t=val_end_t,
    delta_t=DELTA_T,
    window_size=WINDOW_SIZE,
    depth=DEPTH,
    lambda_lasso=None,
    gamma=None,
    n_target=N_TARGET,
    use_sig_y_only=False,
    add_time=True,
)

metrics_y_only = inverse_transform_forecasts(forecast_y_only, y_scaler)
metrics_joint = inverse_transform_forecasts(forecast_joint, y_scaler)

df_y_only = build_forecast_dataframe(forecast_y_only, dates, DELTA_T, metrics_y_only)
df_joint = build_forecast_dataframe(forecast_joint, dates, DELTA_T, metrics_joint)

plot_df = df_y_only[['target_date', 'actual_orig']].merge(
    df_y_only[['target_date', 'forecast_orig']].rename(columns={'forecast_orig': 'signature_y_only'}),
    on='target_date',
    how='left',
).merge(
    df_joint[['target_date', 'forecast_orig']].rename(columns={'forecast_orig': 'signature_joint_path'}),
    on='target_date',
    how='left',
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(plot_df['target_date'], plot_df['actual_orig'], label='Actual', linewidth=2, color='black')
ax.plot(plot_df['target_date'], plot_df['signature_y_only'], label='Signature y-only', linewidth=1.8)
ax.plot(plot_df['target_date'], plot_df['signature_joint_path'], label='Signature joint-path', linewidth=1.8)
ax.set_title('4-week-ahead forecasts for initial claims')
ax.set_xlabel('Target date')
ax.set_ylabel('Initial claims')
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'claims_4_week_forecasts.png', dpi=150, bbox_inches='tight')
plt.show()

plot_df.tail(N_VALIDATION)
